In [1]:
import os
import pandas as pd
import tensorflow as tf
import numpy as np
from datetime import datetime
from working_data import clean_five_minute_data, add_timing, normalize_by_window, clean_hour_data, split_multiresolution_chunks, regression_label_df
from constants.global_constants import *
from modeler import create_regression_model
from regression_losses import compile_model_recommended, compile_model_lightweight

2025-07-23 23:54:49.767418: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-23 23:54:49.804425: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-23 23:54:50.480345: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
try:
    policy = tf.keras.mixed_precision.Policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy(policy)
    print(f"Mixed precision policy set: {policy.name}")
    
    # Verify it's working
    print(f"Compute dtype: {policy.compute_dtype}")  # Should be float16
    print(f"Variable dtype: {policy.variable_dtype}")  # Should be float32
except Exception as e:
    print(f"Could not enable mixed precision: {e}")

Mixed precision policy set: mixed_float16
Compute dtype: float16
Variable dtype: float32


In [3]:
starting_dir = "data/final_data"
working_path = "data/experimenting"
instrument = "GBPUSD#"


In [4]:
df = pd.read_csv(f"{starting_dir}/{instrument}/five_minutes.csv")
print(f"Processing {instrument}...")
print(f"  5 minute Original data: {len(df)} rows")
print(f"  5 minute Time range: {datetime.fromtimestamp(df['time'].min())} to {datetime.fromtimestamp(df['time'].max())}")
df = clean_five_minute_data(df)
print(f"  Cleaned data: {len(df)} rows")
df = add_timing(df)
df = normalize_by_window(
    df, 
    window_size=NORMALIZING_WINDOW_SIZE, 
    low_col='low',
    high_col='high',
    normalizing_cols=[
        'open',
        'high',
        'low',
        'close'
    ],
    label_cols=['open', 'close'])

hour_df = pd.read_csv(f"{starting_dir}/{instrument}/hours.csv")
print(f"  Hour Original data: {len(hour_df)} rows")
print(f"  Hour Time range: {datetime.fromtimestamp(hour_df['time'].min())} to {datetime.fromtimestamp(hour_df['time'].max())}")
hour_df = clean_hour_data(hour_df)
print(f"  Cleaned data: {len(hour_df)} rows")
hour_df = normalize_by_window(
    hour_df, 
    window_size=NORMALIZING_WINDOW_SIZE, 
    low_col='low',
    high_col='high',
    normalizing_cols=[
        'open',
        'high',
        'low',
        'close'
    ],
    label_cols=['open', 'close'])

print(f"Labeling...\n\n\n")
df = regression_label_df(df, window_size=REGRESSION_LABELING_WINDOW_SIZE, 
                positive_slope=POSITIVE_SLOPE, 
                negative_slope=NEGATIVE_SLOPE,
                starting_hour=9,
                ending_hour=18,
                lookback_window=LABEL_LOOKBACK)


os.makedirs(f"{working_path}/{instrument}", exist_ok=True)

hour_df.to_csv(f"{working_path}/{instrument}/hour.csv", index=False)
split_multiresolution_chunks(df_5min=df,
                            df_hour=hour_df,
                            dump_path=f"{working_path}/{instrument}",
                            chunk_size=20000,
                            hour_lookback=OTHER_TOKENS,
                            lookback=NUM_TOKENS,
                            cols=[
                                'time',
                                'open_normalized',
                                'high_normalized',
                                'low_normalized',
                                'close_normalized',
                                'include',
                                'target_high',
                                'target_low'
                            ])

Processing GBPUSD#...
  5 minute Original data: 1709699 rows
  5 minute Time range: 2001-01-01 02:35:00 to 2024-06-01 00:00:00
  Cleaned data: 1702722 rows
  Hour Original data: 145427 rows
  Hour Time range: 2001-01-01 02:00:00 to 2024-06-01 00:00:00
  Cleaned data: 144010 rows
Labeling...





{'total_chunks': 86,
 'total_original_rows': 1702566,
 'chunks_info': [{'chunk_num': 0,
   'start_idx': 0,
   'end_idx': 20000,
   'total_rows': 20000,
   'train_rows': 13999,
   'val_rows': 2936,
   'test_rows': 2937,
   'start_time': 978435600,
   'end_time': 987018600,
   'hour_start_pos': -1},
  {'chunk_num': 1,
   'start_idx': 20000,
   'end_idx': 40000,
   'total_rows': 20000,
   'train_rows': 13999,
   'val_rows': 2936,
   'test_rows': 2937,
   'start_time': 987018900,
   'end_time': 995637000,
   'hour_start_pos': 1560},
  {'chunk_num': 2,
   'start_idx': 40000,
   'end_idx': 60000,
   'total_rows': 20000,
   'train_rows': 13999,
   'val_rows': 2936,
   'test_rows': 2937,
   'start_time': 995637300,
   'end_time': 1004333400,
   'hour_start_pos': 3266},
  {'chunk_num': 3,
   'start_idx': 60000,
   'end_idx': 80000,
   'total_rows': 20000,
   'train_rows': 13999,
   'val_rows': 2936,
   'test_rows': 2937,
   'start_time': 1004333700,
   'end_time': 1012988100,
   'hour_start_pos

In [5]:
import os
from generators.regression_multi_instrument_data_generator import InstrumentConfig, MultiInstrumentDatasetConfig, create_multi_instrument_dataset
from constants.global_constants import FEATURES, NUM_TOKENS, OTHER_TOKENS, BATCH_SIZE, LOOKBACK_WINDOW


instruments = os.listdir(working_path)
instruments = [instrument]
feature_cols = FEATURES

def get_datasets_and_steps(instruments=instruments, working_path=working_path, feature_cols=feature_cols):
    train_instrument_configs = []
    val_instrument_configs = []
    test_instrument_configs = []

    for instrument in instruments:
        train_instrument_configs.append(
            InstrumentConfig(
                name=instrument,
                hourly_data_path=f"{working_path}/{instrument}/hour.csv",
                chunked_data_dir=f"{working_path}/{instrument}/training"
            )
        )
        val_instrument_configs.append(
            InstrumentConfig(
                name=instrument,
                hourly_data_path=f"{working_path}/{instrument}/hour.csv",
                chunked_data_dir=f"{working_path}/{instrument}/validation"
            )
        )
        test_instrument_configs.append(
            InstrumentConfig(
                name=instrument,
                hourly_data_path=f"{working_path}/{instrument}/hour.csv",
                chunked_data_dir=f"{working_path}/{instrument}/testing"
            )
        )

    train_config = MultiInstrumentDatasetConfig(
        instruments=train_instrument_configs,
        main_lookback_tokens=NUM_TOKENS,
        hourly_lookback_tokens=OTHER_TOKENS,
        lookback_window=LOOKBACK_WINDOW,
        batch_size=BATCH_SIZE,
        shuffle_data=True,
        feature_columns=feature_cols,
        max_chunks_per_instrument=25
    )

    val_config = MultiInstrumentDatasetConfig(
        instruments=val_instrument_configs,
        main_lookback_tokens=NUM_TOKENS,
        hourly_lookback_tokens=OTHER_TOKENS,
        lookback_window=LOOKBACK_WINDOW,
        batch_size=BATCH_SIZE,
        shuffle_data=False,
        feature_columns=feature_cols,
        max_chunks_per_instrument=25
    )

    test_config = MultiInstrumentDatasetConfig(
        instruments=test_instrument_configs,
        main_lookback_tokens=NUM_TOKENS,
        hourly_lookback_tokens=OTHER_TOKENS,
        lookback_window=LOOKBACK_WINDOW,
        batch_size=BATCH_SIZE,
        shuffle_data=False,
        feature_columns=feature_cols,
        max_chunks_per_instrument=25
    )


    train_dataset, train_rows = create_multi_instrument_dataset(
        config=train_config,
        repeat_dataset=True
    )
    val_dataset, val_rows = create_multi_instrument_dataset(
        config=val_config,
        repeat_dataset=True
    )
    test_dataset, test_rows = create_multi_instrument_dataset(
        config=test_config,
        repeat_dataset=True
    )

    train_steps = train_rows//BATCH_SIZE
    val_steps = val_rows//BATCH_SIZE
    test_steps = test_rows//BATCH_SIZE

    return (
        (train_dataset, val_dataset, test_dataset),
        (train_steps, val_steps, test_steps)
    )

In [6]:
(train_dataset, val_dataset, test_dataset), (train_steps, val_steps, test_steps) = get_datasets_and_steps()

2025-07-24 00:00:12,814 - INFO - Discovered 86 chunk files for GBPUSD#
2025-07-24 00:00:12,815 - INFO - Loading hourly data for GBPUSD#...
2025-07-24 00:00:12,835 - INFO - Loaded hourly data for GBPUSD#: 143866 rows
2025-07-24 00:00:12,837 - INFO - Applied time threshold for GBPUSD#: 979318800
2025-07-24 00:00:12,838 - INFO - Building indices for GBPUSD#...
2025-07-24 00:00:17,239 - INFO - Built indices for GBPUSD#: 79738 valid samples
2025-07-24 00:00:17,240 - INFO - Building global indices across all instruments...
2025-07-24 00:00:17,246 - INFO - Global indices built: 79738 total samples across 1 instruments
2025-07-24 00:00:17,247 - INFO - === Multi-Instrument Regression Dataset Info ===
2025-07-24 00:00:17,247 - INFO - Total instruments: 1
2025-07-24 00:00:17,248 - INFO -   GBPUSD#: 79738 samples, 86 chunks
2025-07-24 00:00:17,248 - INFO - Global total: 79738 samples
2025-07-24 00:00:17,249 - INFO - Target columns: ['target_high', 'target_low']
2025-07-24 00:00:17,249 - INFO - Shu

In [7]:
def get_naive_baseline_metrics(val_dataset, val_steps):
    """
    Calculate naive baseline metrics for target_high predictions.
    Uses mean prediction as the naive baseline.
    
    Args:
        val_dataset: TensorFlow dataset from create_multi_instrument_dataset
        val_steps: Number of validation steps/batches to process
        
    Returns:
        dict: Contains baseline_value, mae, mse, rmse
    """
    # Collect all target_high values
    all_target_highs = []
    
    for i, batch in enumerate(val_dataset):
        if i >= val_steps:
            break
        (main_input, hourly_input), targets = batch
        target_highs = targets['target_high'].numpy()
        all_target_highs.extend(target_highs)
    
    all_target_highs = np.array(all_target_highs)
    
    # Calculate baseline (mean of all targets)
    baseline_value = np.mean(all_target_highs)
    
    # Create predictions (always predict the mean)
    predictions = np.full_like(all_target_highs, baseline_value)
    
    # Calculate metrics
    mae = np.mean(np.abs(predictions - all_target_highs))
    mse = np.mean((predictions - all_target_highs) ** 2)
    rmse = np.sqrt(mse)
    
    return {
        'baseline_value': baseline_value,
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'total_samples': len(all_target_highs)
    }

In [8]:
get_naive_baseline_metrics(val_dataset, val_steps)

{'baseline_value': 3.4219778,
 'mae': 2.883738,
 'mse': 17.861206,
 'rmse': 4.226252,
 'total_samples': 17088}

In [9]:
model = create_regression_model(d_model=R_D_MODEL, num_heads=R_NUM_HEADS, ff_dim=R_FF_DIM,
                                num_tokens=NUM_TOKENS, other_tokens=OTHER_TOKENS)
model = compile_model_lightweight(model=model)
print(model.metrics_names)

['loss', 'compile_metrics']


In [10]:
history = model.fit(
    train_dataset,
    epochs=50,
    steps_per_epoch=train_steps,
    validation_data=val_dataset,
    validation_steps=val_steps
)

Epoch 1/50


I0000 00:00:1753308055.235346  162149 service.cc:145] XLA service 0x7febf0001f10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1753308055.235383  162149 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 3070 Ti Laptop GPU, Compute Capability 8.6
W0000 00:00:1753308055.467926  162149 random_ops.cc:59] Warning: Using tf.random.uniform with XLA compilation will ignore seeds; consider using tf.random.stateless_uniform instead if reproducible behavior is desired. functional_1/stochastic_gated_transformer_block_1/random_uniform/RandomUniform
2025-07-24 00:00:55.472731: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-07-24 00:00:58.374395: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907
I0000 00:00:1753308074.767062  162374 asm_compiler.cc:369] ptxas warning : Registers are

1245/1245 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step - loss: 12.5184 - target_high_loss: 6.8921 - target_high_mae: 3.8903 - target_high_metric: 0.6963 - target_high_metric_1: 0.1530 - target_high_metric_2: 0.1987 - target_high_mse: 28.2452 - target_low_loss: 5.6263 - target_low_mae: 2.7564 - target_low_metric: 0.4571

I0000 00:00:1753308374.237115  163492 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_41', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1753308374.801869  163505 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_2783', 24 bytes spill stores, 24 bytes spill loads



1245/1245 ━━━━━━━━━━━━━━━━━━━━ 364s 247ms/step - loss: 12.5174 - target_high_loss: 6.8915 - target_high_mae: 3.8901 - target_high_metric: 0.6964 - target_high_metric_1: 0.1530 - target_high_metric_2: 0.1987 - target_high_mse: 28.2412 - target_low_loss: 5.6259 - target_low_mae: 2.7562 - target_low_metric: 0.4571 - val_loss: 11.1395 - val_target_high_loss: 6.1854 - val_target_high_mae: 2.7528 - val_target_high_metric: 0.8277 - val_target_high_metric_1: 0.1104 - val_target_high_metric_2: 0.0192 - val_target_high_mse: 16.7029 - val_target_low_loss: 4.9541 - val_target_low_mae: 2.0386 - val_target_low_metric: 0.4547
Epoch 2/50
1245/1245 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - loss: 10.6259 - target_high_loss: 5.7946 - target_high_mae: 3.5354 - target_high_metric: 0.7230 - target_high_metric_1: 0.2973 - target_high_metric_2: 0.3850 - target_high_mse: 22.5036 - target_low_loss: 4.8313 - target_low_mae: 2.5051 - target_low_metric: 0.4509

I0000 00:00:1753308654.226759  164393 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_41', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1753308656.213041  164391 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_41', 4 bytes spill stores, 4 bytes spill loads



1245/1245 ━━━━━━━━━━━━━━━━━━━━ 279s 224ms/step - loss: 10.6258 - target_high_loss: 5.7945 - target_high_mae: 3.5353 - target_high_metric: 0.7230 - target_high_metric_1: 0.2973 - target_high_metric_2: 0.3850 - target_high_mse: 22.5028 - target_low_loss: 4.8313 - target_low_mae: 2.5051 - target_low_metric: 0.4510 - val_loss: 10.6627 - val_target_high_loss: 6.1589 - val_target_high_mae: 2.7562 - val_target_high_metric: 0.8170 - val_target_high_metric_1: 0.2559 - val_target_high_metric_2: 0.1220 - val_target_high_mse: 16.2897 - val_target_low_loss: 4.6246 - val_target_low_mae: 2.2620 - val_target_low_metric: 0.4616
Epoch 3/50
1245/1245 ━━━━━━━━━━━━━━━━━━━━ 293s 236ms/step - loss: 10.3158 - target_high_loss: 5.5654 - target_high_mae: 3.4753 - target_high_metric: 0.7338 - target_high_metric_1: 0.3111 - target_high_metric_2: 0.4002 - target_high_mse: 21.2184 - target_low_loss: 4.7504 - target_low_mae: 2.5033 - target_low_metric: 0.4544 - val_loss: 10.7853 - val_target_high_loss: 6.1536 - val_